### For GPT2

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers datasets peft accelerate

# Step 2: Imports
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset


# Step 3: Load Model & Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_model.resize_token_embeddings(len(tokenizer))

# Step 4: Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
lora_model = get_peft_model(base_model, lora_config)

# Step 5: Prompt Dataset — this is where you will load the text file containing the data for fine-tuning
#         and populate the `samples` list below.
samples = [
    {"text": "<s> # Write a function to compute factorial of a number\ndef factorial(n):"},
    {"text": "<s> # Write a function to compute the sum of a list\ndef sum_list(lst):"},
    {"text": "<s> # Write a function to reverse a string\ndef reverse_string(s):"},
]
dataset = Dataset.from_list(samples)
tokenized = dataset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=128), batched=False)

# Step 6: Training Arguments
training_args = TrainingArguments(
    output_dir="./lora-gpt2-codegen",
    per_device_train_batch_size=2,
    num_train_epochs=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Step 7: Fine-Tune with LoRA
trainer.train()

# Step 8: Save LoRA model adapter
trainer.model.save_pretrained("./lora-gpt2-codegen")


# ================= Inference & Comparison ============================
# Step 9: Reload Base and LoRA Models for Comparison
from peft import PeftModel

# Base GPT2
base_model = GPT2LMHeadModel.from_pretrained("gpt2").to("cuda").eval()
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load LoRA adapter directly onto base GPT2
base_model_for_lora = GPT2LMHeadModel.from_pretrained("gpt2")
base_model_for_lora.resize_token_embeddings(len(tokenizer))

lora_model = PeftModel.from_pretrained(base_model_for_lora, "./lora-gpt2-codegen")
lora_model.eval().to("cuda")


# Step 10: Define Prompt and Compare
test_prompts = [
    "# Write a function to compute factorial of a number\ndef factorial(n):",

]

def generate_response(model, prompt, max_new_tokens=50):
    inputs = tokenizer("<s> " + prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Step 11: Compare Outputs Before vs After LoRA
for prompt in test_prompts:
    print(f"\n Prompt:\n{prompt}\n")

    print(" Before LoRA:")
    print(generate_response(base_model, prompt))

    print("\n After LoRA:")
    print(generate_response(lora_model, prompt))
    print("-" * 80)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (13 outputs left out)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 Prompt:
# Write a function to compute factorial of a number
def factorial(n):

 Before LoRA:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<s> # Write a function to compute factorial of a number
def factorial(n): return n
def factorial(n): return factorial(n)
def factorial(n): return factorial(n)
def factorial(n): return factorial(n)
def factorial(n): return fact

 After LoRA:
<s> # Write a function to compute factorial of a number
def factorial(n): return n * factorial(n)
def factorial(n): return factorial(n)
def factorial(n): return factorial(n)
def factorial(n): return factorial(n)
def fact
--------------------------------------------------------------------------------


In [ ]:
# Simple Code Prompt
input_text = "<s> # Write a function to compute factorial of a number\ndef factorial(n):"
inputs = tokenizer(input_text, return_tensors="pt", padding=True).to("cuda")
input_ids = inputs.input_ids.to(lora_model.device)
attention_mask = inputs["attention_mask"]

lora_model.eval()
with torch.no_grad():
    outputs = lora_model.generate(input_ids, max_new_tokens=50,  attention_mask=attention_mask, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


<s> # Write a function to compute factorial of a number
def factorial(n): """ Return an integer to compute factorial on """ return factorial(n) if x_index != 0: return factorial(x_index+1) elif x_index != 1: return factorial(x_index+2


### For Pythia, make the following changes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# In Step 3: Load Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-160m")
base_model.resize_token_embeddings(len(tokenizer))


# In Step 4: LoRA configuration's target modules for LoRA are different for Pythia

    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],


# In Step 6: Training Arguments's save path for Pythia should be different than the other model
    output_dir="./lora-pythia-160m-codegen",


# In Step 8: Save LoRA model adapter for Pythia
trainer.model.save_pretrained("./lora-pythia-160m-codegen")


# In Step 9: Reload Base and LoRA Models for Comparison for Pythia
from peft import PeftModel

# Base
base_model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-160m").to("cuda").eval()
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-160m")
tokenizer.pad_token = tokenizer.eos_token

base_model_for_lora = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-160m")
base_model_for_lora.resize_token_embeddings(len(tokenizer))

lora_model = PeftModel.from_pretrained(base_model_for_lora, "./lora-pythia-160m-codegen")

